# Week 7 scratch

Stuff we need to import (copy this to course site)

In [1]:
import csv
import json
import os # operating system library, let's us do things like making new folders
import requests
from time import sleep
from bs4 import BeautifulSoup # the scraping library!

Some example HTML (copy this to course site)

In [2]:
html = """
<html>
  <body>
    <h1>Adeline Ravoux</h1>
    <p class="artist">Vincent van Gogh</p>
    <p class="date">1890</p>
    <a href="/art/1958.31">Artwork page</a>
  </body>
</html>
"""

soup = BeautifulSoup(html, "html.parser")  # html.parser is the built-in parser that comes with python

Having a peek at what's in the soup using the find and find_all methods:

In [12]:
print(soup.find("h1"))
print(soup.find_all("p")) # finds all p elements
print(soup.find("p", class_="date")) # finds the first p element with the class  "date"

<h1>Adeline Ravoux</h1>
[<p class="artist">Vincent van Gogh</p>, <p class="date">1890</p>]
<p class="date">1890</p>


An element's attributes (such as "href", the link URL) are accessed like dictionary keys using square brackets:

In [14]:
link = soup.find("a") # access the first a element
print(link["href"])

/art/1958.31


## Setting up headers (identifying yourself) and fetching a page

(copy this into course site)

In [20]:
headers = {"User-Agent": "INFO664-class-project (educational)"}   # you can call this whatever you want, it shows up on the scraped website's logs as who you are (ie not a Google/AI bot or web browser)

os.makedirs("scraped_pages", exist_ok=True)  # make a new folder to hold our scraped website HTML data. exist_ok = True means, if the directory already exists, do nothing and don't return an error

def fetch_page(url, filename):  # declaring a new function, which takes a URL to be scraped, and a destination filename, as arguments
    if os.path.exists(filename):  # check to see if file already exists, if so, don't download it from the web, load it instead
        with open(filename, "r", encoding="utf-8") as file:
            return file.read()  # in this case, return provides the read file data, and escapes from the function, ignoring everything below

    response = requests.get(url, headers=headers) # grab the file, using our pre-defined headers as the header data

    if not response.ok: # if the response reports an error
        print(url, "failed with status code", response.status_code) # tell us why it failed
        return None # return a "None" object, since there's nothing usable in the response, and escape from this function
    with open(filename, "w",  encoding="utf-8") as file:
        file.write(response.text)

    sleep(10) # wait 10 seconds, since our robots.txt file asks for a 10 second Crawl-delay
    return response.text

In [21]:
url = "https://www.clevelandart.org/art/1966.432"  # URL copied from cleveland museum of art website

html = fetch_page(url, "scraped_pages/cma_1966_432.html")  # grabs the URL using our fetch function
soup = BeautifulSoup(html, "html.parser")  # soups it into a usable object

print(len(html)) # length of characters in HTML  

112467


Finding the title:

text works pretty much the same as [get_text()](https://www.crummy.com/software/BeautifulSoup/bs4/doc/#get-text), except get_text() is configurable 

In [32]:
print(soup.title.get_text())
print(soup.title.text)

Hinds House Window | Cleveland Museum of Art
Hinds House Window | Cleveland Museum of Art


Looking at all elements of a certain type and extracting an attribute (href)

In [42]:
all_links = soup.find_all("a")

for link in all_links:
    print(link.get_text(), link["href"])

Tiffany Glass & Decorating Company /art/collection/search?artists=Tiffany%20Glass%20%26%20Decorating%20Company
America, New York /art/collection/search?culture=America%2C%20New%20York
leaded glass /art/collection/search?medium=leaded%20glass
Gift of Mrs. Robert M. Fallon /art/collection/search?credit=Gift%20of%20Mrs.%20Robert%20M.%20Fallon
CMA's Open Access Initiative. /open-access
209 Tiffany /art/collection/search?gallery=209%20Tiffany
ArtLens App https://cma.org/artlens/artwork/1966.432
Print /print/art/1966.432
Collection /art/collection/search?collection_mapped=Decorative%20Arts
Department /art/collection/search?department=Decorative%20Art%20and%20Design
Type of artwork /art/collection/search?type=Glass
Medium /art/collection/search?medium_mapped=glass
Credit line /art/collection/search?credit=Gift%20of%20Mrs.%20Robert%20M.%20Fallon
Expandable Dragonfly Vase https://shop.clevelandart.org/products/expandable-dragonfly-vase
Louis C. Tiffany Pink Peony Apron https://shop.clevelandart

Having looked at the structure of the HTML file in the browser, we have seen that all the details are stored in a container with an id called `artwork-details-container`. We can have a look at what's in there:

In [49]:
details = soup.find(id="artwork-details-container")
print(len(details))

children = details.find_all(True, recursive=False)  # True means find all elements, recursive means only find the top-level tags of the direct children and not any children thereof)

for index, child in enumerate(children):  # enumerate gives us both the contents of the list item, as well as its index number
    print(index, child.name, "|", child.get_text(" ", strip=True))  # print the index number, the type of element, and the text contents of each child element

14
0 h2 | Details / Information for Hinds House Window
1 div | Hinds House Window
2 div | c. 1900
3 div | maker Tiffany Glass & Decorating Company (America, New York, 1892–1902)
4 div | Culture America, New York
5 div | Medium leaded glass
6 div | Measurements Unframed: 227.3 x 114.3 cm (89 1/2 x 45 in.)
7 div | Credit Line Gift of Mrs. Robert M. Fallon 1966.432
8 div | Public Domain You can copy, modify, and distribute this work, all without asking permission. Learn more about CMA's Open Access Initiative.
9 div | Location 209 Tiffany
10 div | ArtLens App Share Download Print
11 div | ? Did You Know? This window incorporates layered pieces of glass, producing shadows and creating a multidimensional quality.
12 div | Description In this window commissioned for the Howell Hinds house around 1900, thousands of richly colored glass pieces have been arranged and layered to paint an idyllic three-dimensional scene that would have warmly glowed in the afternoon sunlight. Located at 2200 Over

This is mostly good, but some of these elements seem to contain multiple items. Let's have a peek at what's going on in the element at index 3, which seems to have the maker and the biographical info:

In [52]:
print(children[3])

<div class="mt-2 md:mt-4 mb-4"><div class="capitalize">maker</div><div class="flex items-baseline gap-3 flex-wrap justify-start mb-2"><div class="mb-5 md:mb-0 flex-col flex w-full max-w-[120rem] xl:mx-auto !w-auto xl:!mx-0 !mb-0" role="none"><h4 class="!mb-0 first-letter:!normal-case font-fold-grotesque md:mb-4 xsm:mb-2 prose-h4" data-testid="heading" id="-object-object-"><span><a class="!text-blue-03 font-bold" href="/art/collection/search?artists=Tiffany%20Glass%20%26%20Decorating%20Company" target="_self"><span class="underline underline-offset-2">Tiffany Glass &amp; Decorating Company</span></a></span></h4></div><span class="body-1-tight md:body-2-tight text-grey-05 whitespace-nowrap">(America, New York, 1892–1902)</span></div></div>


In [68]:
row = children[3].find_all(True,recursive=False)

for child in row:
    print(child.name, child)  # we can see that the second item (index 1) here has the relevant info

bio_info = row[1].find_all(True,recursive=False) # let's go one level deeper and look at the elements contained therein

for child in bio_info:
    print(child.text)  # this shows the artist name and the bio data as separate entries, let's then put these in their own variables

artist = bio_info[0].get_text()
bio = bio_info[1].get_text()

print(artist,bio)



div <div class="capitalize">maker</div>
div <div class="flex items-baseline gap-3 flex-wrap justify-start mb-2"><div class="mb-5 md:mb-0 flex-col flex w-full max-w-[120rem] xl:mx-auto !w-auto xl:!mx-0 !mb-0" role="none"><h4 class="!mb-0 first-letter:!normal-case font-fold-grotesque md:mb-4 xsm:mb-2 prose-h4" data-testid="heading" id="-object-object-"><span><a class="!text-blue-03 font-bold" href="/art/collection/search?artists=Tiffany%20Glass%20%26%20Decorating%20Company" target="_self"><span class="underline underline-offset-2">Tiffany Glass &amp; Decorating Company</span></a></span></h4></div><span class="body-1-tight md:body-2-tight text-grey-05 whitespace-nowrap">(America, New York, 1892–1902)</span></div>
Tiffany Glass & Decorating Company
(America, New York, 1892–1902)
Tiffany Glass & Decorating Company (America, New York, 1892–1902)


In [69]:
# could also accomplish this with a long traversal of the children in a single line:

print(row)

[<div class="capitalize">maker</div>, <div class="flex items-baseline gap-3 flex-wrap justify-start mb-2"><div class="mb-5 md:mb-0 flex-col flex w-full max-w-[120rem] xl:mx-auto !w-auto xl:!mx-0 !mb-0" role="none"><h4 class="!mb-0 first-letter:!normal-case font-fold-grotesque md:mb-4 xsm:mb-2 prose-h4" data-testid="heading" id="-object-object-"><span><a class="!text-blue-03 font-bold" href="/art/collection/search?artists=Tiffany%20Glass%20%26%20Decorating%20Company" target="_self"><span class="underline underline-offset-2">Tiffany Glass &amp; Decorating Company</span></a></span></h4></div><span class="body-1-tight md:body-2-tight text-grey-05 whitespace-nowrap">(America, New York, 1892–1902)</span></div>]


In [78]:
artist = children[3].find_all("span",recursive=True)[1].get_text()
bio = children[3].find_all("span",recursive=True)[2].get_text()
print(artist)
print(bio)

Tiffany Glass & Decorating Company
(America, New York, 1892–1902)
